In [2]:
%load_ext autoreload
%autoreload 2
%load_ext rpy2.ipython

In [3]:
import ibis
import ibis.selectors as s
import pandas as pd

import src
from src.load import DataLoader

ibis.options.interactive = True
r_colormap = src.r_colormap
r_out = str(src.OUT)
pd.options.display.float_format = "{:.1f}".format

In [4]:
%%R -i r_colormap -i r_out

suppressMessages(library(tidyverse))
library(ggplot2)
library(ggeffects)
library(here)
library(ggpubr)

options(scipen = 999)

cmap <- setNames(r_colormap$color, r_colormap$channel)

here() starts at /Users/lukas/git/ytpop
In addition: Warning message:
In (function (package, help, pos = 2, lib.loc = NULL, character.only = FALSE,  :
  library ‘/nix/store/82m40ksmhhjhcmbfc8n8vip58wbq8cxm-apple-framework-CoreFoundation-11.0.0/Library’ contains no packages


# Load Data

In [5]:
dl = DataLoader()

videos = dl.channels().join(dl.videos(filtered=True), "channel_id").to_pandas()

sentences = (
    dl.sentences(filtered=True)
    .select(~s.cols("tokens"))
    .join(dl.popbert(filtered=False), "sentence_id")
    .to_pandas()
)

sents = sentences.groupby("video_id", observed=True).agg(
    n_sents=("video_id", "size"),
    n_elite=("elite", "sum"),
    n_pplcentr=("pplcentr", "sum"),
    avg_elite=("elite", "mean"),
    avg_pplcentr=("pplcentr", "mean"),
)

# Dataset Summary Table

In [6]:
channel_overview = (
    videos.loc[videos.video_was_live == False]
    .merge(sents, on="video_id", how="left")
    .groupby("channel", observed=True)
    .agg(
        chFollowers=("channel_follower_count", "first"),
        nVideos=("video_was_live", lambda x: (x == 0).sum()),
        nLikesNA=("video_likes", lambda x: x.isna().sum()),
        meanViews=("video_views", "mean"),
        medianViews=("video_views", "median"),
        meanLikes=("video_likes", "mean"),
        medianLikes=("video_likes", "median"),
        meanVideoLen=("video_duration", "mean"),
        nSentences=("n_sents", "sum"),
        # avg_comments=("video_comment_count", lambda x: x.dropna().mean()),
        first_video=("video_datetime_upload", "min"),
        latest_video=("video_datetime_upload", "max"),
    )
)

In [7]:
channel_overview

,chFollowers,nVideos,nLikesNA,meanViews,medianViews,meanLikes,medianLikes,meanVideoLen,nSentences,first_video,latest_video
channel,,,,,,,,,,,
AfD BT,521000,6178,0,53452.5,16467.5,4359.8,2053.0,411.6,346584,2017-12-06 13:23:54,2025-02-20 15:14:38
AfD TV,334000,1808,107,53341.1,19673.0,4363.7,2544.0,565.0,161961,2017-12-08 00:21:22,2025-02-21 15:09:57
CDU,30000,820,1,18450.5,1288.5,187.7,28.0,573.8,68641,2017-12-11 16:28:36,2025-02-22 19:13:53
CSU,6610,171,4,26834.5,930.0,45.4,24.0,438.4,12766,2017-12-14 21:19:06,2025-02-19 13:55:30
FDP,28900,718,717,21020.1,1185.5,105.0,105.0,571.8,54847,2018-01-06 16:02:32,2025-02-22 16:51:16
Greens,35400,638,7,16560.4,1313.5,194.8,32.0,608.2,50822,2018-01-27 10:45:39,2025-02-22 23:09:52
Left,117000,740,3,29378.1,3429.5,1887.4,163.0,590.9,61331,2017-12-11 14:33:45,2025-02-23 16:28:05
SPD,34500,1040,2,10682.7,2434.0,193.5,85.0,601.2,87155,2017-12-07 12:17:56,2025-02-23 20:14:54


In [22]:
inlines = src.OUT / "manuscript/inlines"
inlines.mkdir(exist_ok=True)

In [ ]:
# number of broken transcripts

broken_transcripts = dl.broken_transcripts(filtered=True).to_pandas()
n_broken = len(broken_transcripts)

n_broken = f"{n_broken:,.0f}"
(inlines / "n_broken.txt").write_text(n_broken)
print(f"faulty transcripts: {n_broken}")

faulty transcripts: 80


In [ ]:
# sum of durations

sum_of_seconds = videos.video_duration.sum()
hour_duration = f"{round(sum_of_seconds / 60 / 60, 1):,.1f}"
(inlines / "sum_duration.txt").write_text(hour_duration)
print(f"Total sum of video durations: {hour_duration} hours")

Total sum of video durations: 1,658.6 hours


In [ ]:
# number of videos

count_videos = len(videos)
count_videos = f"{count_videos:,.0f}"
(inlines / "n_videos.txt").write_text(count_videos)
print(f"Total number of valid videos: {count_videos}")

Total number of valid videos: 12,113


In [ ]:
# number of sentencs

count_sents = len(sentences)
count_sents = f"{count_sents:,.0f}"
(inlines / "n_sents.txt").write_text(count_sents)
print(f"Total number of valid sentences: {count_sents}")

Total number of valid sentences: 844,107


In [12]:
summary_table = channel_overview.drop(["first_video", "latest_video"], axis=1).T

summary_table

channel,AfD BT,AfD TV,CDU,CSU,FDP,Greens,Left,SPD
chFollowers,521000.0,334000.0,30000.0,6610.0,28900.0,35400.0,117000.0,34500.0
nVideos,6178.0,1808.0,820.0,171.0,718.0,638.0,740.0,1040.0
nLikesNA,0.0,107.0,1.0,4.0,717.0,7.0,3.0,2.0
meanViews,53452.5,53341.1,18450.5,26834.5,21020.1,16560.4,29378.1,10682.7
medianViews,16467.5,19673.0,1288.5,930.0,1185.5,1313.5,3429.5,2434.0
meanLikes,4359.8,4363.7,187.7,45.4,105.0,194.8,1887.4,193.5
medianLikes,2053.0,2544.0,28.0,24.0,105.0,32.0,163.0,85.0
meanVideoLen,411.6,565.0,573.8,438.4,571.8,608.2,590.9,601.2
nSentences,346584.0,161961.0,68641.0,12766.0,54847.0,50822.0,61331.0,87155.0


In [13]:
path = src.OUT / "tables/dataset_summary.csv"
path.unlink(missing_ok=True)
summary_table.to_csv(path)

# View Count Violin Plot

In [14]:
df = videos.merge(sents, on="video_id")

In [19]:
%%R -i df -w 1000 -h 800

df_plot <- df %>%
   mutate(
      likes = video_likes + 1,
      views = video_views + 1,
)

view_plot = ggplot(df_plot, aes(x=channel, y=views, fill=channel)) +
   geom_boxplot(alpha=0.6) +
   geom_violin(alpha=0.3, trim=T, scale="width") +
   scale_y_continuous(trans=scales::log10_trans(), breaks=scales::breaks_log(n=8)) +
   scale_color_manual(values=cmap, aesthetics=c("color", "fill")) +
   theme_ggeffects(
      base_family = "serif",
      base_size = 21
   ) +
   theme(
      axis.text.x=element_text(angle=20, hjust=1),
      legend.position = "none"
   ) +
   xlab("Channel") +
   ylab("log10(ViewCount)")

like_plot = ggplot(df_plot, aes(x=channel, y=likes, fill=channel)) +
   geom_boxplot(alpha=0.6) +
   geom_violin(alpha=0.3, trim=T, scale="width") +
   scale_y_continuous(trans=scales::log10_trans(), breaks=scales::breaks_log(n=8)) +
   scale_color_manual(values=cmap, aesthetics=c("color", "fill")) +
   theme_ggeffects(
      base_family = "serif",
      base_size = 21
   ) +
   theme(
      axis.text.x=element_text(angle=20, hjust=1),
      legend.position = "none"
   ) +
   xlab("Channel") +
   ylab("log10(LikeCount)")


ggarrange(view_plot, like_plot, ncol=2)

p <- here(r_out, "/figures/view_count.svg")
if (file.exists(p)) file.remove(p)
ggsave(p, width=13.9, height=11.4)

In addition: Warning messages:
1: Removed 841 rows containing non-finite outside the scale range
(`stat_boxplot()`). 
2: Removed 841 rows containing non-finite outside the scale range
(`stat_ydensity()`). 
3: Groups with fewer than two datapoints have been dropped.
ℹ Set `drop = FALSE` to consider such groups for position adjustment purposes. 


# Populism Amount Plot

In [21]:
%%R -i df -w 1000 -h 800

df_plot <- df %>%
   mutate(
      elite = (n_elite / n_sents * 100) + 1,
      pplcentr = (n_pplcentr / n_sents * 100) + 1,
)

elite_plot = ggplot(df_plot, aes(x=channel, y=elite, fill=channel)) +
   geom_boxplot(alpha=0.6, outliers=F, coef=0.5) +
   scale_color_manual(values=cmap, aesthetics=c("color", "fill")) +
   theme_ggeffects(
      base_family = "serif",
      base_size = 22
   ) +
   theme(
      axis.text.x=element_text(angle=20, hjust=1),
      legend.position = "none"
   ) +
   xlab("Channel") +
   ylab("% Anti-Elitism")

pplcentr_plot = ggplot(df_plot, aes(x=channel, y=pplcentr, fill=channel)) +
   geom_boxplot(alpha=0.6, outliers=F, coef=0.5) +
   scale_color_manual(values=cmap, aesthetics=c("color", "fill")) +
   theme_ggeffects(
      base_family = "serif",
      base_size = 22
   ) +
   theme(
      axis.text.x=element_text(angle=20, hjust=1),
      legend.position = "none"
   ) +
   xlab("Channel") +
   ylab("% People-Centrism")


ggarrange(elite_plot, pplcentr_plot, ncol=2)

p <- here(r_out, "/figures/populism_per_party.svg")
if (file.exists(p)) file.remove(p)
ggsave(p, width=13.9, height=11.0)